# AtaRehber: evaluation on a Colab TPU

Runs `python -m evals.run_eval` with `Qwen/Qwen3-8B` served by [vLLM on TPU](https://docs.vllm.ai/projects/tpu/en/latest/)
(`LLM_BACKEND=vllm`), once per entry of `VARIANTS`, saves the reports to Google Drive, and compares them in the
last cell.

**Before running:**
* **Runtime → Change runtime type → TPU v6e-1.** Qwen3-8B in bf16 takes about 16 GB; a v5e chip has 16 GB of
  memory and cannot hold it with its context.
* **Secrets** (key icon on the left): `GITHUB_TOKEN` with read access to the repository, *Notebook access* on.

The TPU serves the language model only; the embedding and reranker models run on the CPU of the TPU machine.
A TPU can be opened by one process at a time: the vLLM server holds it, so no other cell may use it.

In [ ]:
# Settings
REPO = "https://github.com/SirAlper/AtaRehber.git"
BRANCH = "main"
MODEL = "Qwen/Qwen3-8B"       # Hugging Face name of the model vLLM serves
# Settings to compare (names of .env.example). The model runs once per entry; {} is the default behaviour
# (qwen3 thinks for answers, compliance verdicts, and combined answers: OLLAMA_THINK_TASKS).
VARIANTS = [{}]               # e.g. [{}, {"OLLAMA_THINK_TASKS": "none"}, {"OLLAMA_THINK_TASKS": "answer,compliance,synthesis,grader"}]
# Context window of the server in tokens: prompt plus reply must fit, or the request fails (a thinking call is
# then asked again without thinking). 16384 leaves room for the thinking of long compliance contexts.
MAX_MODEL_LEN = 16384
DATASET = "university"        # "university" (127 questions, laws and regulations) or "demo" (51 questions)
STAGES = "all"                # "retrieval", "retrieval,routing", "all", "agents", "grader"
IDS = ""                      # e.g. "u-rek-01,u-rek-02" for a quick trial; empty: every question
# Reports are saved here (Google Drive survives the session); empty keeps them in /content/results
DRIVE_DIR = "/content/drive/MyDrive/AtaRehber"

In [ ]:
import sys
print(sys.version)
if sys.version_info < (3, 12):
    print("WARNING: the project is tested on Python 3.12+; this image is older and may fail.")
!ls /dev/vfio /dev/accel* 2>/dev/null || echo "No TPU found: choose Runtime > Change runtime type > TPU v6e-1"

## 1. Code and Python packages

The project's packages go into the notebook's Python; vLLM gets its own environment (`/content/vllm-env`), so
its pinned versions of torch, transformers, and JAX do not clash with the project's.

In [ ]:
%cd /content
from google.colab import userdata
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = ""
clone_url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
!rm -rf AtaRehber && git clone --quiet --depth 1 --branch {BRANCH} {clone_url} AtaRehber
%cd /content/AtaRehber
# The token is not kept in the clone
!git remote set-url origin {REPO}
!git log --oneline -1
!pip install -q -r requirements.txt
# The embedding and reranker models need PyTorch; a TPU image may come without it (the CPU build is enough)
try:
    import torch
    print("torch", torch.__version__)
except ImportError:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
!pip install -q uv
!uv venv -q --python 3.12 /content/vllm-env
!uv pip install -q --python /content/vllm-env/bin/python vllm-tpu
# Which TPU is there: Qwen3-8B (bf16) needs a v6e chip (32 GB); the check runs in its own process, which releases
# the TPU when it ends
!/content/vllm-env/bin/python -c "import jax; d = jax.devices()[0]; print(d.device_kind, '| chips:', jax.device_count())"

## 2. The vLLM server

Starting takes several minutes: the model is downloaded (about 16 GB) and compiled for the TPU. The log is in
`/content/vllm.log`.

In [ ]:
import subprocess, time, urllib.request

serve = [
    "/content/vllm-env/bin/vllm", "serve", MODEL,
    "--port", "8000",
    "--max-model-len", str(MAX_MODEL_LEN),
    # qwen3's thinking comes apart from the reply (reasoning_content)
    "--reasoning-parser", "qwen3",
    # Tool calls of custom agents and DOC_AGENT_TOOLS=true
    "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
    # The evaluation asks one question at a time; fewer parallel sequences compile faster
    "--max-num-seqs", "8",
]
server = subprocess.Popen(serve, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
started = time.time()
while True:
    if server.poll() is not None:
        !tail -40 /content/vllm.log
        raise RuntimeError("The vLLM server stopped; see the log above")
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        print(f"vLLM is serving {MODEL} after {time.time() - started:.0f} s")
        break
    except Exception:
        if time.time() - started > 45 * 60:
            !tail -40 /content/vllm.log
            raise RuntimeError("The vLLM server did not start within 45 minutes")
        time.sleep(10)

## 3. Embedding and reranker models (about 3.3 GB)

In [ ]:
!python download_model.py 2>&1 | tail -5

## 4. Evaluation

Each question prints a line when it starts and one when it is done, with the progress and the time left.

In [ ]:
import os

if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
results = f"{DRIVE_DIR}/results" if DRIVE_DIR else "/content/results"
os.makedirs(results, exist_ok=True)

base_env = dict(os.environ)
args = f"--stages {STAGES} --out {results}"
if DATASET == "university":
    args += " --dataset evals/dataset_university.jsonl --corpus evals/corpus_university --no-database"
if IDS:
    args += f" --ids {IDS}"
for number, variant in enumerate(VARIANTS, 1):
    os.environ.clear()
    os.environ.update(base_env)
    os.environ.update(
        {
            "LLM_BACKEND": "vllm",
            "VLLM_BASE_URL": "http://localhost:8000/v1",
            "VLLM_MODEL": MODEL,
            "RAG_DEVICE": "cpu",
            # The vLLM server holds the TPU; PyTorch in this process must not try to open it
            "PJRT_DEVICE": "CPU",
        }
    )
    if DATASET == "university":
        # The threshold measured for article-aware chunks (docs/evaluation.md)
        os.environ["RAG_MIN_RERANKER_SCORE"] = "0.05"
    os.environ.update({key: str(value) for key, value in variant.items()})
    print(f"\n######## Run {number}/{len(VARIANTS)}: {MODEL} {variant or '(default settings)'} ########")
    print("python -m evals.run_eval", args)
    !python -m evals.run_eval {args} 2>&1 | grep -v -E "^\[20|Batches|Loading weights"

## 5. The reports side by side

In [ ]:
import glob, json

paths = sorted(glob.glob(f"{results}/report_*.json"))[-len(VARIANTS):]
if not paths:
    raise SystemExit("No report was written; see the output above.")
reports = [json.load(open(path, encoding="utf-8")) for path in paths]
labels = [f"run {number}" for number in range(1, len(reports) + 1)]
for label, path, report, variant in zip(labels, paths, reports, VARIANTS[-len(reports):]):
    settings = report["config"]
    print(f"{label}: {settings['llm_model']} via {settings.get('llm_backend')} {variant or '(default settings)'}"
          f" | thinking for: {', '.join(settings.get('ollama_think_tasks') or []) or 'none'}")
    print(f"   commit {report['git_commit']} | {report['case_count']} cases | {path}")

width = 12
for stage in reports[0]["summary"]:
    print(f"\n== {stage} ==")
    print(" " * 36 + "".join(f"{label:>{width}}" for label in labels))
    for key, value in reports[0]["summary"][stage].items():
        if isinstance(value, (dict, list)):
            continue
        values = [report["summary"].get(stage, {}).get(key) for report in reports]
        shown = [f"{v:.3f}" if isinstance(v, float) else str(v) for v in values]
        print(f"  {key:<34}" + "".join(f"{text:>{width}}" for text in shown))